# Experimento MTLHate Prompt
## Validación del enriquecimiento MTL en el prompt del chatbot de ciberacoso

Valida si añadir información MTL (tipo de discurso, grupo afectado, intensidad) al prompt
mejora significativamente las respuestas del SLM respecto al pipeline V1.

**Hipótesis**: El enriquecimiento del prompt con tipo de discurso, grupo afectado e intensidad
produce respuestas más específicas y clínicamente adecuadas que el prompt base V1.

**Modelo evaluado**: Gemma 7B (Ollama)  
**Casos de prueba**: 10 mensajes con etiquetas MTL asignadas manualmente  
**Combinaciones totales**: 2 variantes × 10 casos = 20 respuestas

## Diseño Experimental

### Variantes comparadas

| Variante | Técnica | Hipótesis |
|:---:|---|---|
| **V1** | Prompt base (emoción + RAG) — equivalente a `builder.py` | H_base: la emoción detectada y el contexto RAG son suficientes para una respuesta adecuada |
| **V2** | Prompt enriquecido con bloque de análisis MTL simulado | H_MTL: añadir tipo de discurso, grupo afectado e intensidad produce respuestas más específicas y clínicamente adecuadas |

### Criterios de evaluación (rúbrica manual, Sección 5)

- **Especificidad**: ¿V2 menciona el grupo afectado (homofobia, racismo…) con mayor precisión?
- **Tono**: ¿V2 ajusta el tono de forma más adecuada al tipo de discurso detectado?
- **Failsafe de intensidad**: ¿Se activa el aviso de emergencia en casos de intensidad ≥ 5 (casos 3, 5 y 10)?
- **Hope amplification**: ¿V2 amplifica la narrativa de resiliencia en mensajes de tipo `hope` (caso 9)?

### Condiciones de validez

- Contexto RAG idéntico en V1 y V2 para aislar el efecto exclusivo del bloque MTL
- Temperatura seleccionada automáticamente según la emoción detectada (`TEMPERATURE_BY_EMOTION`)
- Etiquetas MTL asignadas manualmente (sin clasificador real en producción)

## 0. Imports y configuración

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
import ollama
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from src.rag import RAGRetriever
from src.prompts.builder_v1 import BASE_SYSTEM_PROMPT, EMOTION_VARIANTS, TEMPERATURE_BY_EMOTION

print(f"Directorio de trabajo: {Path.cwd()}")

# Verificar que Ollama está en ejecución
print("\nComprobando conexión con Ollama...")
try:
    _resp = ollama.list()
    nombres_modelos = [m.model for m in _resp.models]
    print(f"Ollama activo. Modelos disponibles: {nombres_modelos}")
    if not any("gemma" in n for n in nombres_modelos):
        raise RuntimeError(
            "gemma:7b no encontrado en Ollama.\n"
            "Ejecuta primero: ollama pull gemma:7b"
        )
    print("gemma:7b disponible")
except Exception as e:
    raise RuntimeError(
        f"Ollama no está disponible o gemma:7b no instalado.\n"
        f"Error: {e}\n"
        "Inicia Ollama con: ollama serve"
    )


## 1. Casos de Prueba

10 mensajes representativos con etiquetas MTL asignadas manualmente (sin clasificador real).
Cubren diferentes emociones, tipos de discurso, grupos afectados e intensidades,
incluyendo casos límite como amenaza de muerte (intensidad 6) y discurso esperanzador.

In [ ]:
casos = [
    {
        "id": 1,
        "user_msg": "me están insultando por ser gay en los grupos de clase",
        "emotion": "fear",
        "emotion_conf": 0.74,
        "mtl_type": "hate",
        "mtl_group": "homofobia",
        "mtl_intensity": 3
    },
    {
        "id": 2,
        "user_msg": "me llaman gordo todos los días en el insta, ya no quiero ir al insti",
        "emotion": "sadness",
        "emotion_conf": 0.81,
        "mtl_type": "hate",
        "mtl_group": "gordofobia",
        "mtl_intensity": 3
    },
    {
        "id": 3,
        "user_msg": "me están amenazando con publicar fotos mías si no hago lo que dicen",
        "emotion": "fear",
        "emotion_conf": 0.89,
        "mtl_type": "hate",
        "mtl_group": "misoginia",
        "mtl_intensity": 5
    },
    {
        "id": 4,
        "user_msg": "no aguanto más los comentarios racistas que me mandan por discord",
        "emotion": "anger",
        "emotion_conf": 0.76,
        "mtl_type": "hate",
        "mtl_group": "racismo",
        "mtl_intensity": 3
    },
    {
        "id": 5,
        "user_msg": "me han dicho que me van a pegar a la salida del instituto",
        "emotion": "fear",
        "emotion_conf": 0.92,
        "mtl_type": "hate",
        "mtl_group": "ninguno",
        "mtl_intensity": 5
    },
    {
        "id": 6,
        "user_msg": "estoy muy mal, me siento sola y no sé por qué",
        "emotion": "sadness",
        "emotion_conf": 0.68,
        "mtl_type": "none",
        "mtl_group": "ninguno",
        "mtl_intensity": 0
    },
    {
        "id": 7,
        "user_msg": "me están insultando mucho pero no sé si es para tanto",
        "emotion": "others",
        "emotion_conf": 0.45,
        "mtl_type": "offensive",
        "mtl_group": "ninguno",
        "mtl_intensity": 2
    },
    {
        "id": 8,
        "user_msg": "me acosan por trans en todas las redes, ya no puedo más",
        "emotion": "sadness",
        "emotion_conf": 0.83,
        "mtl_type": "hate",
        "mtl_group": "transfobia",
        "mtl_intensity": 4
    },
    {
        "id": 9,
        "user_msg": "creo que estoy empezando a salir adelante, hoy he bloqueado a todos",
        "emotion": "joy",
        "emotion_conf": 0.61,
        "mtl_type": "hope",
        "mtl_group": "ninguno",
        "mtl_intensity": 0
    },
    {
        "id": 10,
        "user_msg": "llevan semanas difundiendo fotos mías y diciéndome que me mate",
        "emotion": "fear",
        "emotion_conf": 0.95,
        "mtl_type": "hate",
        "mtl_group": "misoginia",
        "mtl_intensity": 6
    }
]

print(f"{len(casos)} casos de prueba cargados.")


## 2. Definición de Variantes

### V1 — Sin información MTL
Implementación equivalente a `src/prompts/builder.py`. Usa emoción detectada y contexto RAG.

### V2 — Con información MTL simulada
Añade al system prompt un bloque de análisis del discurso **antes** del contexto RAG,
con tipo, grupo afectado e intensidad cualitativa. Activa failsafe para intensidades
críticas (≥ 5) y amplificación de narrativa para discurso esperanzador (`hope`).

In [ ]:
def build_prompt_v1(emotion: str, emotion_conf: float, rag_context: str) -> list[dict]:
    """
    Construye el prompt V1 (sin información MTL), equivalente a builder.py.

    Args:
        emotion: Emoción detectada ('fear', 'sadness', 'anger', etc.)
        emotion_conf: Confianza del clasificador (0.0-1.0)
        rag_context: Texto de los chunks RAG recuperados

    Returns:
        Lista de mensajes formato chat [{"role": "system", "content": ...}]
    """
    effective_emotion = emotion if emotion_conf >= 0.4 else "others"
    system = BASE_SYSTEM_PROMPT + "\n\n" + EMOTION_VARIANTS[effective_emotion]
    if rag_context:
        system += (
            "\n\nCONTEXTO CLÍNICO (usa esta información para orientar tu "
            "respuesta si es relevante, sin reproducirla literalmente):\n"
            + rag_context
        )
    return [{"role": "system", "content": system}]


print("✓ build_prompt_v1 definida")


In [ ]:
_INTENSIDAD_DESC: dict[int, str] = {
    0: "sin contenido de odio",
    1: "desacuerdo",
    2: "trato negativo",
    3: "insultos al carácter",
    4: "demonización",
    5: "incitación a la violencia",
    6: "amenaza de muerte",
}


def build_prompt_v2_simulated(
    emotion: str,
    emotion_conf: float,
    rag_context: str,
    mtl_type: str,
    mtl_group: str,
    mtl_intensity: int,
) -> list[dict]:
    """
    Construye el prompt V2 enriquecido con análisis MTL simulado.

    Añade al system prompt un bloque de análisis del discurso antes del
    contexto RAG: tipo de discurso, grupo afectado, intensidad e instrucciones
    adicionales según el nivel de riesgo o la presencia de discurso esperanzador.

    Args:
        emotion: Emoción detectada
        emotion_conf: Confianza del clasificador (0.0-1.0)
        rag_context: Texto de los chunks RAG recuperados
        mtl_type: Tipo de discurso ('hate', 'offensive', 'hope', 'none')
        mtl_group: Grupo afectado (ej. 'homofobia', 'racismo', 'ninguno')
        mtl_intensity: Intensidad del discurso de odio (0-6)

    Returns:
        Lista de mensajes formato chat [{"role": "system", "content": ...}]
    """
    effective_emotion = emotion if emotion_conf >= 0.4 else "others"
    system = BASE_SYSTEM_PROMPT + "\n\n" + EMOTION_VARIANTS[effective_emotion]

    # Bloque MTL insertado antes del contexto RAG
    desc_intensidad = _INTENSIDAD_DESC.get(mtl_intensity, "desconocida")
    grupo_display = mtl_group if mtl_group != "ninguno" else "no identificado"

    mtl_block = (
        "\n\nANÁLISIS DEL DISCURSO DETECTADO:\n"
        f"- Tipo: {mtl_type}  (hate / offensive / hope / none)\n"
        f"- Grupo afectado: {grupo_display}\n"
        f"- Intensidad: {mtl_intensity}/6  ({desc_intensidad})"
    )

    if mtl_intensity >= 5:
        mtl_block += (
            "\n\u26a0\ufe0f ALERTA: Intensidad crítica detectada. Prioriza la derivación "
            "a recursos de emergencia: 024, ANAR 900 202 010, 112."
        )

    if mtl_type == "hope":
        mtl_block += (
            "\nEl usuario expresa resiliencia o esperanza. Amplifica esta narrativa."
        )

    system += mtl_block

    if rag_context:
        system += (
            "\n\nCONTEXTO CLÍNICO (usa esta información para orientar tu "
            "respuesta si es relevante, sin reproducirla literalmente):\n"
            + rag_context
        )

    return [{"role": "system", "content": system}]


print(" build_prompt_v2_simulated definida")

# Verificación rápida: el bloque MTL aparece antes del contexto RAG
_preview = build_prompt_v2_simulated(
    "fear", 0.9, "[RAG placeholder]", "hate", "homofobia", 3
)[0]["content"]
mtl_pos = _preview.find("ANÁLISIS DEL DISCURSO")
rag_pos  = _preview.find("CONTEXTO CLÍNICO")
assert mtl_pos < rag_pos, "ERROR: bloque MTL debe preceder al contexto RAG"
print(f"Orden correcto — MTL en pos {mtl_pos}, RAG en pos {rag_pos}")


## 3. Ejecución

Para cada caso se generan dos respuestas con `gemma:7b` (num_predict=300):
- **V1**: prompt sin información MTL
- **V2**: prompt con bloque de análisis MTL simulado

El contexto RAG es idéntico en ambas variantes para garantizar una comparación justa.
La temperatura se selecciona automáticamente según la emoción detectada.

In [ ]:
# Cargar el retriever FAISS
retriever = RAGRetriever()
print(f"✓ RAGRetriever cargado — {retriever.index.ntotal} chunks en el índice")


def invocar_llm(mensajes: list[dict], temperatura: float) -> str:
    """Convierte mensajes a formato LangChain e invoca el SLM."""
    role_map = {"system": SystemMessage, "user": HumanMessage, "assistant": AIMessage}
    lc_msgs = [role_map[m["role"]](content=m["content"]) for m in mensajes]
    _llm = ChatOllama(model="gemma:7b", temperature=temperatura, num_predict=300)
    return _llm.invoke(lc_msgs).content


# Ejecutar el experimento
resultados = []
total = len(casos)
print(f"\nEjecutando experimento: {total} casos x 2 variantes (V1 sin MTL / V2 con MTL)...\n")

for i, caso in enumerate(casos, 1):
    print(f"[{i}/{total}] Caso {caso['id']}: {caso['user_msg'][:55]}...")
    emotion      = caso["emotion"]
    emotion_conf = caso["emotion_conf"]
    user_msg     = caso["user_msg"]
    temperatura  = TEMPERATURE_BY_EMOTION.get(emotion, 0.6)

    # Contexto RAG idéntico para V1 y V2
    rag_chunks  = retriever.retrieve(query=user_msg, emotion=emotion, top_k=3)
    rag_context = "\n---\n".join(c.chunk.content for c in rag_chunks) if rag_chunks else ""

    # Respuesta V1 (sin MTL)
    msgs_v1 = build_prompt_v1(emotion, emotion_conf, rag_context)
    msgs_v1.append({"role": "user", "content": user_msg})
    resp_v1 = invocar_llm(msgs_v1, temperatura)

    # Respuesta V2 (con MTL simulado)
    msgs_v2 = build_prompt_v2_simulated(
        emotion, emotion_conf, rag_context,
        caso["mtl_type"], caso["mtl_group"], caso["mtl_intensity"],
    )
    msgs_v2.append({"role": "user", "content": user_msg})
    resp_v2 = invocar_llm(msgs_v2, temperatura)

    resultados.append({
        "id":            caso["id"],
        "user_msg":      user_msg,
        "emotion":       emotion,
        "mtl_type":      caso["mtl_type"],
        "mtl_group":     caso["mtl_group"],
        "mtl_intensity": caso["mtl_intensity"],
        "respuesta_v1":  resp_v1,
        "respuesta_v2":  resp_v2,
    })
    print(f"    ✓ V1: {len(resp_v1)} chars | V2: {len(resp_v2)} chars")

print(f"\n✓ Experimento completado: {len(resultados)}/{total} casos procesados")


In [ ]:
df = pd.DataFrame(resultados)
print(f"DataFrame con {len(df)} casos y {len(df.columns)} columnas:\n")
display(
    df[["id", "emotion", "mtl_type", "mtl_group", "mtl_intensity"]]
    .style
    .set_caption("Resumen de casos — experimento MTLHate")
    .hide(axis="index")
)


## 4. Visualización Comparativa

Para cada caso se muestran en paralelo la respuesta V1 (sin MTL) y V2 (con MTL simulado),
con cabecera que indica emoción, grupo afectado e intensidad.

In [ ]:
SEP = "─" * 60

for _, row in df.iterrows():
    print(SEP)
    print(
        f"CASO {row['id']} | emoción: {row['emotion']} "
        f"| grupo: {row['mtl_group']} | intensidad: {row['mtl_intensity']}"
    )
    print(f"USUARIO: {row['user_msg']}")
    print(SEP)
    print("V1 (sin MTL):")
    print(row["respuesta_v1"])
    print()
    print("V2 (con MTL simulado):")
    print(row["respuesta_v2"])
    print(SEP)
    print()


## 5. Rúbrica de Evaluación Manual

**Aspectos a evaluar:**
- **Especificidad**: ¿V2 menciona el grupo afectado (homofobia, racismo…) con mayor precisión?
- **Tono**: ¿V2 ajusta el tono de forma más adecuada al tipo de discurso?
- **Failsafe**: ¿Se activa correctamente el aviso de emergencia en casos 3, 5 y 10 (intensidad ≥ 5)?
- **Hope amplification**: ¿V2 amplifica la narrativa de resiliencia en el caso 9?

| Caso | ¿Mejor V1 o V2? | ¿En qué aspecto? | ¿Diferencia relevante? |
|:----:|:-----------------:|------------------|:---------------------:|
| 1    | V1/V2 |  El mismo texto escrito de otra forma  | No |
| 2    | V2 | Más concreto al tipo de acoso y una respuesta mejor construida | Sí |
| 3    | V1 | Se adapta mejor a la gravedad del mensaje | Sí |
| 4    | V1 | Ambas respuestas están equivocadas, no son conscientes de ataques racistas, se creen que el grupo es homofobía y añaden Instagram | No |
| 5    | V1 | Mejor respues ante la situación, pero ambnos se comportan de manera parecida | No |
| 6    | V2 | Mejor construcción de la respuesta | Sí |
| 7    | V1/V2 | Tiene una respuesta más centrada en el mensaje, pero exagera el problema de lo que realmente es cuando V2 presenta unas mejores opciones de apoyo | No |
| 8    | V1 | Mejor respuesta en general, ni V1 ni V2 comentan amenzas sobre la transfobia | No |
| 9    | V2 | Capta mejor el mensaje de esperanza que V1, ya que V1 lo describe como un momento difícil y ofrece apoyo con otra actividad | Sí |
| 10   | V2 | Ambos tienen un mismo comportamiento en la respuesta, V2 puede ser algo más correcta | No |

**Puntuación global:**
- Casos en que V2 es claramente mejor: 3 / 10
- Casos en que no hay diferencia apreciable: 6 / 10
- Casos en que V1 es igual o mejor: 7 / 10

## 6. Conclusión


**1. ¿En qué porcentaje de casos V2 produce una respuesta claramente mejor?**

> En el 30 % de los casos V2 produce una respuesta claramente mejor

---

**2. ¿En qué tipos de mensajes (con/sin mención explícita del grupo) se observa mayor diferencia?**

> La mayor diferencia se observa en mensajes con baja intensidad o con un mensaje esperanzador, dando una respuesta más adecuada para este caso, ya que se alegra de escuchar esa situación con un mensaje más esperanzador, sin describir que es un momento díficil, y le ofrece realizar otras actividades para que el usuario se sienta mejor y olvide por lo que ha pasado.

---

**3. ¿El coste de implementar el clasificador MTLHate real justifica la mejora observada?**

> Pienso que por la latencia adicional (~50-100 ms por inferencia), complejidad de integración y la leve mejora que ofrece y en ocasiones imperceptible, no merece la pena su implementación.

---

**4. Decisión final:**

- Descartar e incluir como trabajo futuro en la memoria del TFG

> En conclusión, el clasificador MTLHate para un chatbot de apoyo contra el ciberacoso como un rol enrutador hacia el RAG no obtiene la suficiente importancia para que merezca la pena. Es posible que la intesidad del mensaje sea un dato importante a la hora de que el SLM contruya su mensaje acuerdo con el tono del usario, pero realmente, en la mayoría de casos nos encontraremos con mensajes de alta gravedad que V1 maneja ya bien. Además, gracias al sistema RAG implementado, V1 llega a detectar en varios casos el grupo amenazado por la similitud coseno, siendo una función que genera mayor latencia si añadimos un clasificador MTLHate. Por lo tanto, mejorando el sistema RAG, concretando casos, refinando los chunks que ya tenemos, eliminando chunks innecesarios o hacerlos más generalistas, puede ya generar un gran salto de calidad del chatbot sin añadirle complejidad.